# Assignment 3: Model Training and Fine-Tuning for Legal Article Relevance (Adilet Search)

**Task.** Given a user's question and an article of Kazakhstan legislation, decide whether the article answers the question. The search system uses this model as its **reranker**: BM25 proposes 20 candidate articles, and the model reorders them. So every model here is scored twice, on the same test set:

- as a **classifier** of (question, article) pairs: F1 and ROC-AUC;
- as a **reranker**: nDCG@10, MRR@10 and Recall@10 of the correct article.

This notebook applies the recipe of our earlier IMDb experiment (tuned classical models → transfer learning with a frozen encoder → full fine-tuning → LoRA) to the project's own data, in Russian and Kazakh.

| Rubric criterion | Where |
|---|---|
| Compare ≥ 3 models on the same split and metrics | §9 unified table (9 rows) |
| Hyperparameter tuning: search space + best parameters | §3 BM25 grid, §5 LogReg / SVM grid search, §7 learning-rate search |
| Fine-tuning: lr, batch, epochs, optimizer, loss | §7 |
| Transfer learning: what transfers, why this base | §6 |
| LoRA for an LLM: r, alpha, dropout, targets, trainable parameters, why PEFT | §8 |
| Classical ML optimisation, before / after | §3 and §5 |
| Unified results table, ≥ 2 shared metrics | §9 |
| Scientific interpretation | §10 |
| Reproducibility: seeds, versions, configs | §0, and `a3_results/` |

**Run on a GPU runtime** (Colab: Runtime → Change runtime type → T4 GPU), then *Run all*. About 40 minutes on a T4. No tokens or logins are needed.

In [ ]:
# §0 Setup. The project repository is public; the corpus is downloaded from Zenodo and verified.
import os, subprocess, sys

REPO_URL = "https://github.com/R0onen/adilet-search.git"
REPO_REF = os.environ.get("ADILET_REF", "main")
if os.path.exists("ml/scripts/download_data.py"):  # already inside the repository
    REPO_DIR = os.getcwd()
elif os.path.exists("../scripts/download_data.py"):  # opened from ml/notebooks
    REPO_DIR = os.path.abspath("../..")
else:
    REPO_DIR = os.path.abspath("adilet-search")
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, REPO_DIR], check=True)

# Colab already has torch, transformers, scikit-learn and pandas; these are the extras.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bm25s==0.3.13", "PyStemmer==3.1.0", "peft", "pyarrow"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", os.path.join(REPO_DIR, "ml")], check=True)
sys.path.insert(0, os.path.join(REPO_DIR, "ml", "src"))
print("repository:", REPO_DIR)

In [ ]:
import gc, json, math, random, time
from pathlib import Path

import bm25s
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import peft
import sklearn
import torch
import transformers
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

from adilet_ml.retrieval.sparse import tokenize  # the product's tokenizer (Russian stemming)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
transformers.set_seed(SEED)

# SMOKE=1 shrinks everything for a CPU check of the code; the reported numbers need SMOKE=0.
SMOKE = os.environ.get("SMOKE", "0") == "1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
REPO = Path(REPO_DIR)
OUT = Path("a3_results"); OUT.mkdir(exist_ok=True)

print(f"python {sys.version.split()[0]} | torch {torch.__version__} | transformers {transformers.__version__} | "
      f"peft {peft.__version__} | scikit-learn {sklearn.__version__} | pandas {pd.__version__} | bm25s {bm25s.__version__}")
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU (training will be slow)")
print("seed:", SEED, "| smoke run:", SMOKE)

## §1 Data

- **Corpus:** AdiletCodex v1.0 (CC BY 4.0, doi:10.5281/zenodo.22812626), the official texts parsed from adilet.zan.kz. We use the 8 main codes in Russian and Kazakh: 9,259 articles. The project script downloads the file and checks its MD5.
- **Questions** (`data/eval/` in the repository):
  - *synthetic*: written by an LLM for sampled articles, two per article: `plain` (everyday wording) and `term` (legal terminology). These are the closest we have to real user questions.
  - *title*: the official title of an article, used as a query for it. Only used to enlarge the **training** set.
- **Label:** the article a question was written for is relevant (1); every other article is not (0).
- **Split:** by article (the Russian and Kazakh versions of an article are always in the same subset), fixed in `data/eval/splits.json`, seed 42. Validation and test use **synthetic questions only**.

In [ ]:
if not (REPO / "data/processed/chunks.parquet").exists():
    subprocess.run([sys.executable, str(REPO / "ml/scripts/download_data.py")], check=True)

chunks = pd.read_parquet(REPO / "data/processed/chunks.parquet",
                         columns=["article_id", "lang", "chunk_index", "text", "unit_title", "doc_short_title"])
articles = (chunks.sort_values(["article_id", "chunk_index"])
            .groupby("article_id", sort=False)
            .agg(lang=("lang", "first"), body=("text", "\n".join), title=("unit_title", "first"), doc=("doc_short_title", "first"))
            .reset_index())
articles["title"] = articles["title"].fillna("")
# What the models read: the code's name, then the article body. The title is left out on purpose:
# title queries would otherwise be answered by string matching.
articles["passage"] = articles["doc"] + ". " + articles["body"].str.slice(0, 2000)
corpus = {lang: g.reset_index(drop=True) for lang, g in articles.groupby("lang")}
print({lang: len(g) for lang, g in corpus.items()}, "articles")

def read_jsonl(path):
    return pd.DataFrame([json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line])

synthetic = read_jsonl(REPO / "data/eval/synthetic/queries.jsonl").merge(read_jsonl(REPO / "data/eval/synthetic/qrels.jsonl"), on="query_id")
titles = read_jsonl(REPO / "data/eval/queries.jsonl").merge(read_jsonl(REPO / "data/eval/qrels.jsonl"), on="query_id")
title_split = {qid: split for split, ids in json.loads((REPO / "data/eval/splits.json").read_text(encoding="utf-8"))["query_splits"].items() for qid in ids}
titles["split"] = titles["query_id"].map(title_split)
titles["query_type"] = "title"

N_TITLE_TRAIN = 200 if SMOKE else 2500
title_train = titles[titles["split"] == "train"].sample(N_TITLE_TRAIN, random_state=SEED)
queries = pd.concat([synthetic, title_train], ignore_index=True)[["query_id", "text", "lang", "query_type", "split", "article_id"]]
if SMOKE:
    queries = queries.groupby(["split", "lang"]).head(8).reset_index(drop=True)

print(queries.groupby(["split", "query_type", "lang"]).size().unstack(fill_value=0))
display(queries[queries["query_type"] != "title"].groupby("query_type").head(2)[["text", "lang", "query_type"]])

## §2 Candidates and metrics

For each question, BM25 retrieves the **top 20** articles of the same language. Those 20 are what every model scores:

- **training pairs:** the correct article (label 1) plus the 7 highest-ranked wrong ones (label 0, "hard negatives": articles that share words with the question but are not the answer);
- **evaluation pairs:** all 20 candidates, for validation and test questions.

**Metrics** (the same for every model, on the same test pairs):

| Metric | What it shows | Why it fits |
|---|---|---|
| nDCG@10 | is the correct article near the top (position-discounted) | users read the top of the list; primary metric |
| MRR@10 | how soon the correct article appears | one correct article per question |
| Recall@10 | is it on the first page at all | it must be there to be read or cited |
| F1 (positive class), ROC-AUC | quality of the relevance decision itself | pair classification view; accuracy is not used because only 1 pair in 20 is positive |

If BM25 misses the correct article in its top 20, no reranker can recover it. That **ceiling** is printed below.

In [ ]:
K_CAND, N_NEG = 20, 7

def bm25_tokens(texts, lang):
    return [tokenize(t, lang) for t in texts]

corpus_tokens = {lang: bm25_tokens(c["body"], lang) for lang, c in corpus.items()}
article_pos = {lang: {a: i for i, a in enumerate(c["article_id"])} for lang, c in corpus.items()}

def bm25_index(lang, k1=1.2, b=0.75):
    index = bm25s.BM25(k1=k1, b=b)
    index.index(corpus_tokens[lang], show_progress=False)
    return index

def retrieve(index, lang, texts, k=K_CAND):
    docs, scores = index.retrieve(bm25_tokens(texts, lang), k=k, show_progress=False)
    return docs, scores

def rank_metrics(ranks):
    # rank of the correct article (1-based; inf = not among the candidates)
    ranks = np.asarray(ranks, dtype=float)
    hit = ranks <= 10
    return {"nDCG@10": float(np.where(hit, 1 / np.log2(ranks + 1), 0).mean()),
            "MRR@10": float(np.where(hit, 1 / ranks, 0).mean()),
            "Recall@10": float(hit.mean())}

assert abs(rank_metrics([1, 2, np.inf])["nDCG@10"] - (1 + 1 / math.log2(3)) / 3) < 1e-9
print("metric check OK")

## §3 Classical optimisation 1: BM25, default vs tuned

BM25 has two hyperparameters: `k1` (how much repeated words count) and `b` (how strongly long articles are penalised). **Search space:** `k1` ∈ {0.6, 0.9, 1.2, 1.5, 2.0} × `b` ∈ {0.3, 0.5, 0.75, 0.9}, chosen per language on the **validation** questions by nDCG@10.

In [ ]:
def bm25_ranks(index, lang, frame):
    docs, _ = retrieve(index, lang, frame["text"].tolist())
    target = frame["article_id"].map(article_pos[lang]).to_numpy()
    hits = docs == target[:, None]
    return np.where(hits.any(1), hits.argmax(1) + 1, np.inf)

val_q = queries[queries["split"] == "val"]
grid_rows, best_bm25 = [], {}
for lang in ("ru", "kk"):
    frame = val_q[val_q["lang"] == lang]
    for k1 in [0.6, 0.9, 1.2, 1.5, 2.0]:
        for b in [0.3, 0.5, 0.75, 0.9]:
            grid_rows.append({"lang": lang, "k1": k1, "b": b, **rank_metrics(bm25_ranks(bm25_index(lang, k1, b), lang, frame))})
grid = pd.DataFrame(grid_rows)
for lang in ("ru", "kk"):
    best = grid[grid["lang"] == lang].sort_values("nDCG@10", ascending=False).iloc[0]
    default = grid[(grid["lang"] == lang) & (grid["k1"] == 1.2) & (grid["b"] == 0.75)].iloc[0]
    best_bm25[lang] = (float(best["k1"]), float(best["b"]))
    print(f"{lang}: best k1={best['k1']}, b={best['b']} (val nDCG@10 {best['nDCG@10']:.4f}); default 1.2 / 0.75: {default['nDCG@10']:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, lang in zip(axes, ("ru", "kk")):
    table = grid[grid["lang"] == lang].pivot(index="k1", columns="b", values="nDCG@10")
    image = ax.imshow(table.values, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(table.columns)), table.columns); ax.set_yticks(range(len(table.index)), table.index)
    ax.set_xlabel("b (length normalisation)"); ax.set_ylabel("k1 (term saturation)")
    ax.set_title(f"BM25 validation nDCG@10, {'Russian' if lang == 'ru' else 'Kazakh'}")
    for i in range(table.shape[0]):
        for j in range(table.shape[1]):
            ax.text(j, i, f"{table.values[i, j]:.3f}", ha="center", va="center", color="white", fontsize=8)
    fig.colorbar(image, ax=ax)
plt.tight_layout(); plt.show()

In [ ]:
# Candidate pairs from the tuned BM25: this table is the input of every model below.
def build_pairs(frame, n_neg=None):
    rows = []
    for lang in ("ru", "kk"):
        part = frame[frame["lang"] == lang]
        if part.empty:
            continue
        docs, scores = retrieve(bm25_index(lang, *best_bm25[lang]), lang, part["text"].tolist())
        ids = corpus[lang]["article_id"].to_numpy()
        for (_, q), cand, score in zip(part.iterrows(), docs, scores):
            kept_neg = 0
            for rank, (d, s) in enumerate(zip(cand, score), 1):
                label = int(ids[d] == q["article_id"])
                if n_neg is not None and not label:
                    if kept_neg >= n_neg:
                        continue
                    kept_neg += 1
                rows.append((q["query_id"], q["text"], lang, q["query_type"], int(d), float(s), rank, label))
            if n_neg is not None and q["article_id"] not in ids[cand]:  # training always sees the positive
                rows.append((q["query_id"], q["text"], lang, q["query_type"], article_pos[lang][q["article_id"]], 0.0, K_CAND + 1, 1))
    pairs = pd.DataFrame(rows, columns=["query_id", "query", "lang", "query_type", "doc", "bm25", "bm25_rank", "label"])
    pairs["passage"] = [corpus[l]["passage"].iat[d] for l, d in zip(pairs["lang"], pairs["doc"])]
    return pairs

train_pairs = build_pairs(queries[queries["split"] == "train"], n_neg=N_NEG)
val_pairs = build_pairs(queries[queries["split"] == "val"])
test_pairs = build_pairs(queries[queries["split"] == "test"])
for name, p in (("train", train_pairs), ("val", val_pairs), ("test", test_pairs)):
    print(f"{name}: {len(p):,} pairs, {p['query_id'].nunique():,} questions, {p['label'].mean():.1%} positive")
ceiling = test_pairs.groupby("query_id")["label"].max().mean()
print(f"Ceiling: the correct article is among the {K_CAND} BM25 candidates for {ceiling:.1%} of the test questions")

In [ ]:
# One evaluation function for every model: scores for the test pairs -> classifier + ranking metrics.
RESULTS, SCORES = [], {}

def evaluate(name, family, scores, pairs=None, threshold=None, extra=None):
    pairs = test_pairs if pairs is None else pairs
    frame = pairs.assign(score=np.asarray(scores, dtype=float))
    order = frame.sort_values(["query_id", "score"], ascending=[True, False])
    order["rank"] = order.groupby("query_id").cumcount() + 1
    ranks = order[order["label"] == 1].set_index("query_id")["rank"].reindex(frame["query_id"].unique()).fillna(np.inf)
    row = {"model": name, "family": family, **rank_metrics(ranks.to_numpy())}
    if threshold is not None:
        pred = (frame["score"] >= threshold).astype(int)
        row.update({"F1": f1_score(frame["label"], pred), "Precision": precision_score(frame["label"], pred, zero_division=0),
                    "Recall": recall_score(frame["label"], pred), "ROC-AUC": roc_auc_score(frame["label"], frame["score"])})
    row.update(extra or {})
    if pairs is test_pairs:
        RESULTS[:] = [r for r in RESULTS if r["model"] != name] + [row]
        SCORES[name] = frame["score"].to_numpy()
    return row

def best_threshold(scores, labels):
    # the decision threshold is chosen on validation pairs, never on test
    grid = np.quantile(scores, np.linspace(0.5, 0.995, 60))
    return float(max(grid, key=lambda t: f1_score(labels, scores >= t)))

# BM25 itself as a ranker: default parameters, then tuned
default_rows = []
for lang in ("ru", "kk"):
    part = queries[(queries["split"] == "test") & (queries["lang"] == lang)]
    default_rows.append(bm25_ranks(bm25_index(lang), lang, part))
RESULTS.append({"model": "BM25 (default k1=1.2, b=0.75)", "family": "classical", **rank_metrics(np.concatenate(default_rows))})
print(evaluate("BM25 (tuned)", "classical", test_pairs["bm25"]))

## §4 Features for the classical models

Classical models cannot read a pair of texts, so each (question, article) pair is described by 8 numbers:

| Feature | Meaning |
|---|---|
| `bm25`, `bm25_rr` | BM25 score and 1 / BM25 rank |
| `tfidf_word`, `tfidf_char` | cosine similarity of word and character n-gram TF-IDF vectors (character n-grams tolerate inflection, which matters for Kazakh) |
| `coverage`, `jaccard` | share of the question's words found in the article; word-set overlap |
| `len_doc`, `len_query` | log lengths |

All TF-IDF statistics are fitted on the article texts only, never on questions or labels (no leakage).

In [ ]:
vectorizers = {}
for lang, c in corpus.items():
    word = TfidfVectorizer(tokenizer=lambda t, lang=lang: tokenize(t, lang), token_pattern=None, lowercase=False,
                           sublinear_tf=True, min_df=2, dtype=np.float32)
    char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True, min_df=3, max_features=200_000, dtype=np.float32)
    vectorizers[lang] = (word, word.fit_transform(c["body"]), char, char.fit_transform(c["body"].str.slice(0, 3000)))
token_sets = {lang: [set(t) for t in toks] for lang, toks in corpus_tokens.items()}

FEATURES = ["bm25", "bm25_rr", "tfidf_word", "tfidf_char", "coverage", "jaccard", "len_doc", "len_query"]

def add_features(pairs):
    out = pairs.copy()
    for column in FEATURES[2:]:
        out[column] = 0.0
    out["bm25_rr"] = 1.0 / out["bm25_rank"]
    for lang in ("ru", "kk"):
        mask = (out["lang"] == lang).to_numpy()
        if not mask.any():
            continue
        word, word_matrix, char, char_matrix = vectorizers[lang]
        part = out[mask]
        unique = part["query"].drop_duplicates().tolist()
        position = {q: i for i, q in enumerate(unique)}
        q_index = part["query"].map(position).to_numpy()
        docs = part["doc"].to_numpy()
        q_word, q_char = word.transform(unique), char.transform(unique)
        out.loc[mask, "tfidf_word"] = np.asarray(q_word[q_index].multiply(word_matrix[docs]).sum(1)).ravel()
        out.loc[mask, "tfidf_char"] = np.asarray(q_char[q_index].multiply(char_matrix[docs]).sum(1)).ravel()
        q_sets = [set(tokenize(q, lang)) for q in unique]
        inter = np.array([len(q_sets[i] & token_sets[lang][d]) for i, d in zip(q_index, docs)])
        q_len = np.array([max(len(q_sets[i]), 1) for i in q_index])
        union = np.array([max(len(q_sets[i] | token_sets[lang][d]), 1) for i, d in zip(q_index, docs)])
        out.loc[mask, "coverage"] = inter / q_len
        out.loc[mask, "jaccard"] = inter / union
        out.loc[mask, "len_doc"] = np.log1p([len(corpus_tokens[lang][d]) for d in docs])
        out.loc[mask, "len_query"] = np.log1p(q_len)
    return out

train_f, val_f, test_f = add_features(train_pairs), add_features(val_pairs), add_features(test_pairs)
display(train_f.groupby("label")[FEATURES].mean().round(3))

## §5 Classical optimisation 2: Logistic Regression and Linear SVM

- **Baseline:** Logistic Regression with default parameters (`C = 1`, no class weights).
- **Tuned:** grid search with 3-fold **GroupKFold** (all pairs of a question stay in one fold), scored by ROC-AUC.

| Model | Search space |
|---|---|
| Logistic Regression | `C` ∈ {0.01, 0.1, 1, 10, 100}; `class_weight` ∈ {none, balanced} |
| Linear SVM | `C` ∈ {0.01, 0.1, 1, 10}; `class_weight` ∈ {none, balanced} |

In [ ]:
X_train, y_train, groups = train_f[FEATURES], train_f["label"], train_f["query_id"]

def classical(name, model, decision):
    val_scores, test_scores = decision(model, val_f[FEATURES]), decision(model, test_f[FEATURES])
    threshold = best_threshold(val_scores, val_f["label"].to_numpy())
    return evaluate(name, "classical", test_scores, threshold=threshold)

proba = lambda m, X: m.predict_proba(X)[:, 1]
margin = lambda m, X: m.decision_function(X)

baseline_lr = Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression(max_iter=2000, random_state=SEED))]).fit(X_train, y_train)
print(classical("TF-IDF features + LogReg (baseline)", baseline_lr, proba))

cv = GroupKFold(n_splits=3)
lr_search = GridSearchCV(
    Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression(max_iter=2000, random_state=SEED))]),
    {"clf__C": [0.01, 0.1, 1, 10, 100], "clf__class_weight": [None, "balanced"]},
    scoring="roc_auc", cv=cv, n_jobs=-1).fit(X_train, y_train, groups=groups)
print("LogReg best parameters:", lr_search.best_params_, "| CV ROC-AUC", round(lr_search.best_score_, 4))
print(classical("TF-IDF features + LogReg (tuned)", lr_search.best_estimator_, proba))

svm_search = GridSearchCV(
    Pipeline([("scale", StandardScaler()), ("clf", LinearSVC(random_state=SEED, max_iter=5000))]),
    {"clf__C": [0.01, 0.1, 1, 10], "clf__class_weight": [None, "balanced"]},
    scoring="roc_auc", cv=cv, n_jobs=-1).fit(X_train, y_train, groups=groups)
print("Linear SVM best parameters:", svm_search.best_params_, "| CV ROC-AUC", round(svm_search.best_score_, 4))
print(classical("TF-IDF features + Linear SVM (tuned)", svm_search.best_estimator_, margin))

coefficients = pd.Series(lr_search.best_estimator_["clf"].coef_[0], index=FEATURES).sort_values()
coefficients.plot.barh(figsize=(7, 3), color=["#c44e52" if v < 0 else "#4c72b0" for v in coefficients])
plt.title("Tuned Logistic Regression: standardised feature weights"); plt.xlabel("coefficient"); plt.tight_layout(); plt.show()

## §6 Transfer learning: a pretrained cross-encoder

A **cross-encoder** reads the question and the article together and outputs one relevance score.

**Base model:** `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` (118 M parameters).

- *What it already knows:* the multilingual MiniLM encoder was pretrained on text in about 100 languages, then trained for passage relevance on mMARCO: MS MARCO web-search questions translated into 14 languages, **including Russian, not Kazakh**. So general "does this passage answer this question" knowledge transfers; Kazakh transfers only through the multilingual encoder.
- *What it does not know:* legal vocabulary, the style of statutes, and how an everyday question maps to a provision.
- *Why this base:* it is multilingual, it is small enough to rerank 20 articles on a CPU within the product's 2-second budget, and its licence (Apache 2.0) allows our use.

Two transfer settings, as in our IMDb experiment:

1. **Zero-shot:** use the model as it is, no training on our data.
2. **Frozen encoder:** freeze all encoder weights and train only the small classification head on our pairs (feature extraction).

In [ ]:
from torch.utils.data import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments

CE_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
MAX_LEN = 128 if SMOKE else 320
FP16 = DEVICE == "cuda"

class PairDataset(Dataset):
    # (question, article) pairs tokenised together, truncating only the article
    def __init__(self, pairs, tokenizer, max_len=MAX_LEN, float_labels=True):
        self.encoded = tokenizer(pairs["query"].tolist(), pairs["passage"].tolist(), truncation="only_second", max_length=max_len)
        self.labels = pairs["label"].astype("float32" if float_labels else "int64").to_numpy()
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        return {**{k: v[i] for k, v in self.encoded.items()}, "labels": self.labels[i]}

class BCETrainer(Trainer):
    # One output logit, binary cross-entropy loss (the standard cross-encoder objective)
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(outputs.logits.view(-1), labels.float().view(-1))
        return (loss, outputs) if return_outputs else loss

def count_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

def predict_scores(trainer, dataset, binary_logit=True):
    logits = trainer.predict(dataset).predictions
    logits = logits[0] if isinstance(logits, tuple) else logits
    return logits.reshape(-1) if binary_logit else logits[:, 1] - logits[:, 0]

def training_args(name, lr, epochs, batch=32, **kw):
    return TrainingArguments(output_dir=f"runs/{name}", learning_rate=lr, num_train_epochs=epochs,
                             per_device_train_batch_size=batch, per_device_eval_batch_size=128,
                             warmup_steps=int(0.1 * math.ceil(len(train_pairs) / batch) * epochs), weight_decay=0.01, fp16=FP16, seed=SEED, logging_steps=50,
                             save_strategy="no", report_to="none", disable_tqdm=True, **kw)

ce_tokenizer = AutoTokenizer.from_pretrained(CE_NAME)
ce_train = PairDataset(train_pairs, ce_tokenizer)
ce_val, ce_test = PairDataset(val_pairs, ce_tokenizer), PairDataset(test_pairs, ce_tokenizer)
collator = DataCollatorWithPadding(ce_tokenizer)

def evaluate_transformer(name, family, trainer, val_set, test_set, binary_logit=True, extra=None):
    val_scores = predict_scores(trainer, val_set, binary_logit)
    threshold = best_threshold(val_scores, val_pairs["label"].to_numpy())
    val_row = evaluate(name, family, val_scores, pairs=val_pairs)
    row = evaluate(name, family, predict_scores(trainer, test_set, binary_logit), threshold=threshold,
                   extra={"val nDCG@10": val_row["nDCG@10"], **(extra or {})})
    return row

# 1. Zero-shot: no training at all
zero_model = AutoModelForSequenceClassification.from_pretrained(CE_NAME).to(DEVICE)
zero_trainer = BCETrainer(model=zero_model, args=training_args("zero", 0.0, 0), data_collator=collator)
trainable, total = count_parameters(zero_model)
print(evaluate_transformer("Cross-encoder, zero-shot (transfer, no training)", "transfer", zero_trainer, ce_val, ce_test,
                           extra={"trainable params": 0, "total params": total}))

In [ ]:
# 2. Frozen encoder: only the classification head is trained
frozen_model = AutoModelForSequenceClassification.from_pretrained(CE_NAME).to(DEVICE)
for parameter in frozen_model.base_model.parameters():
    parameter.requires_grad = False
trainable, total = count_parameters(frozen_model)
print(f"Frozen encoder: {trainable:,} trainable of {total:,} parameters ({trainable / total:.3%})")

FROZEN = {"learning_rate": 1e-3, "epochs": 1 if SMOKE else 3, "batch_size": 32, "optimizer": "AdamW", "loss": "binary cross-entropy"}
frozen_trainer = BCETrainer(model=frozen_model, args=training_args("frozen", FROZEN["learning_rate"], FROZEN["epochs"]),
                            train_dataset=ce_train, data_collator=collator)
started = time.time(); frozen_trainer.train()
print(evaluate_transformer("Cross-encoder, frozen encoder + trained head", "transfer", frozen_trainer, ce_val, ce_test,
                           extra={"trainable params": trainable, "total params": total, "train minutes": (time.time() - started) / 60}))
del zero_model, zero_trainer, frozen_model, frozen_trainer; gc.collect(); torch.cuda.empty_cache()

## §7 Full fine-tuning of the cross-encoder

All weights are updated on our training pairs.

| Setting | Value |
|---|---|
| Loss | binary cross-entropy on one output logit |
| Optimizer | AdamW, weight decay 0.01, linear warm-up over the first 10% of steps, then linear decay |
| Batch size | 32 |
| Epochs | 2 |
| Max length | 320 tokens (question + start of the article) |
| Precision | mixed (fp16) on GPU |
| **Search space** | learning rate ∈ {1e-5, 2e-5, 5e-5}, chosen by validation nDCG@10 |
| Hard negatives | 7 per question, from the BM25 top 20 |

The model with the best validation nDCG@10 is kept and saved to `a3_results/reranker-ft/`: this is the reranker for the product.

In [ ]:
LEARNING_RATES = [2e-5] if SMOKE else [1e-5, 2e-5, 5e-5]
FT = {"epochs": 1 if SMOKE else 2, "batch_size": 32, "optimizer": "AdamW (weight_decay 0.01, warmup 10%)", "loss": "binary cross-entropy", "max_len": MAX_LEN}
search, best = [], None
for lr in LEARNING_RATES:
    transformers.set_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(CE_NAME).to(DEVICE)
    trainer = BCETrainer(model=model, args=training_args(f"ft-{lr}", lr, FT["epochs"]), train_dataset=ce_train, data_collator=collator)
    started = time.time(); trainer.train(); minutes = (time.time() - started) / 60
    val_ndcg = evaluate("val", "ft", predict_scores(trainer, ce_val), pairs=val_pairs)["nDCG@10"]
    search.append({"learning_rate": lr, "val nDCG@10": val_ndcg, "train minutes": minutes})
    print(search[-1])
    if best is None or val_ndcg > best[0]:
        best = (val_ndcg, lr, minutes)
        trainer.save_model(str(OUT / "reranker-ft")); ce_tokenizer.save_pretrained(str(OUT / "reranker-ft"))
    del model, trainer; gc.collect(); torch.cuda.empty_cache()

display(pd.DataFrame(search).round(4))
FT["learning_rate"] = best[1]
print("Best fine-tuning configuration:", FT)

ft_model = AutoModelForSequenceClassification.from_pretrained(str(OUT / "reranker-ft")).to(DEVICE)
ft_trainer = BCETrainer(model=ft_model, args=training_args("ft-eval", 0.0, 0), data_collator=collator)
trainable, total = count_parameters(ft_model)
print(evaluate_transformer("Cross-encoder, fully fine-tuned", "fine-tuned", ft_trainer, ce_val, ce_test,
                           extra={"trainable params": trainable, "total params": total, "train minutes": best[2]}))
del ft_model, ft_trainer; gc.collect(); torch.cuda.empty_cache()

## §8 LoRA: adapting an LLM with 0.1% of its parameters

**Base model:** `Qwen/Qwen2.5-0.5B-Instruct`, a small multilingual LLM (494 M parameters), used here as a pair classifier: it reads "question + article" and a 2-class head decides relevant / not relevant.

**Why PEFT (LoRA) instead of full fine-tuning:**

- Full fine-tuning updates all 494 M weights. It needs far more GPU memory (weights + gradients + optimizer state), and with our few thousand questions it overfits easily.
- LoRA freezes the model and learns small low-rank matrices added to chosen layers. Only about 0.1% of the parameters train, the adapter file is a few MB, and the same base model can serve several adapters.

| LoRA setting | Value | Meaning |
|---|---|---|
| `r` | 8 | rank of the added matrices (capacity of the adapter) |
| `lora_alpha` | 16 | scaling of the update (`alpha / r` = 2) |
| `lora_dropout` | 0.05 | regularisation |
| `target_modules` | `q_proj`, `v_proj` | the query and value projections of every attention layer |
| Learning rate, epochs, batch | 2e-4, 2, 16 | adapters need a higher learning rate than full fine-tuning |

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

LLM_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
LORA = {"r": 8, "lora_alpha": 16, "lora_dropout": 0.05, "target_modules": ["q_proj", "v_proj"],
        "learning_rate": 2e-4, "epochs": 1 if SMOKE else 2, "batch_size": 16, "max_len": 128 if SMOKE else 320}

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm_tokenizer.padding_side = "right"
llm_tokenizer.truncation_side = "right"
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

class LlmPairDataset(Dataset):
    def __init__(self, pairs, max_len):
        texts = [f"Вопрос: {q}\nСтатья: {p}" for q, p in zip(pairs["query"], pairs["passage"])]
        self.encoded = llm_tokenizer(texts, truncation=True, max_length=max_len)
        self.labels = pairs["label"].astype("int64").to_numpy()
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        return {**{k: v[i] for k, v in self.encoded.items()}, "labels": int(self.labels[i])}

class WeightedTrainer(Trainer):
    # 1 positive per 7 negatives: weight the positive class so the model does not learn "always no"
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        weight = torch.tensor([1.0, float(N_NEG)], device=outputs.logits.device, dtype=outputs.logits.dtype)
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=weight)
        return (loss, outputs) if return_outputs else loss

llm = AutoModelForSequenceClassification.from_pretrained(LLM_NAME, num_labels=2)
llm.config.pad_token_id = llm_tokenizer.pad_token_id
lora_config = LoraConfig(task_type=TaskType.SEQ_CLS, r=LORA["r"], lora_alpha=LORA["lora_alpha"],
                         lora_dropout=LORA["lora_dropout"], target_modules=LORA["target_modules"])
llm = get_peft_model(llm, lora_config).to(DEVICE)
llm.print_trainable_parameters()
trainable, total = count_parameters(llm)
print("LoRA configuration:", LORA)

llm_train = LlmPairDataset(train_pairs, LORA["max_len"])
llm_val, llm_test = LlmPairDataset(val_pairs, LORA["max_len"]), LlmPairDataset(test_pairs, LORA["max_len"])
llm_trainer = WeightedTrainer(model=llm, args=training_args("lora", LORA["learning_rate"], LORA["epochs"], batch=LORA["batch_size"]),
                              train_dataset=llm_train, data_collator=DataCollatorWithPadding(llm_tokenizer))
started = time.time(); llm_trainer.train()
print(evaluate_transformer("Qwen2.5-0.5B + LoRA", "LoRA (LLM)", llm_trainer, llm_val, llm_test, binary_logit=False,
                           extra={"trainable params": trainable, "total params": total, "train minutes": (time.time() - started) / 60}))
llm.save_pretrained(str(OUT / "qwen-lora-adapter"))
del llm, llm_trainer; gc.collect(); torch.cuda.empty_cache()

## §9 Unified results (same test questions, same candidates, same metrics)

In [ ]:
results = pd.DataFrame(RESULTS)
columns = ["model", "family", "nDCG@10", "MRR@10", "Recall@10", "F1", "ROC-AUC", "trainable params", "train minutes"]
results = results.reindex(columns=[c for c in columns if c in results.columns])
print(f"Test set: {test_pairs['query_id'].nunique():,} synthetic questions, {len(test_pairs):,} pairs. "
      f"Reranking ceiling (correct article among the {K_CAND} candidates): {ceiling:.1%}")
display(results.round(4))
results.to_csv(OUT / "results.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
plot = results.set_index("model")[["nDCG@10", "MRR@10", "Recall@10"]]
plot.plot.barh(ax=axes[0]); axes[0].invert_yaxis(); axes[0].axvline(ceiling, color="gray", ls="--", lw=1)
axes[0].set_title("Ranking metrics on the test questions (dashed: ceiling)"); axes[0].set_xlabel("score"); axes[0].set_ylabel("")
classifier_rows = results.dropna(subset=["F1"]).set_index("model")[["F1", "ROC-AUC"]]
classifier_rows.plot.barh(ax=axes[1]); axes[1].invert_yaxis()
axes[1].set_title("Pair classification on the test pairs"); axes[1].set_xlabel("score"); axes[1].set_ylabel("")
plt.tight_layout(); plt.savefig(OUT / "results.png", dpi=150); plt.show()

In [ ]:
# Breakdown by language and by question type: where does each kind of model help?
def breakdown(name):
    frame = test_pairs.assign(score=SCORES[name]).sort_values(["query_id", "score"], ascending=[True, False])
    frame["rank"] = frame.groupby("query_id").cumcount() + 1
    meta = test_pairs.drop_duplicates("query_id").set_index("query_id")[["lang", "query_type"]]
    ranks = frame[frame["label"] == 1].set_index("query_id")["rank"].reindex(meta.index).fillna(np.inf)
    rows = {}
    for column in ("lang", "query_type"):
        for value, ids in meta.groupby(column).groups.items():
            rows[f"{column}={value}"] = rank_metrics(ranks.loc[ids].to_numpy())["nDCG@10"]
    return rows

picked = [m for m in ["BM25 (tuned)", "TF-IDF features + LogReg (tuned)", "Cross-encoder, zero-shot (transfer, no training)",
                      "Cross-encoder, fully fine-tuned", "Qwen2.5-0.5B + LoRA"] if m in SCORES]
by_group = pd.DataFrame({m: breakdown(m) for m in picked}).T
display(by_group.round(3))
by_group.T.plot.bar(figsize=(11, 4), rot=0); plt.ylabel("nDCG@10"); plt.title("Test nDCG@10 by language and question type")
plt.legend(fontsize=8); plt.tight_layout(); plt.savefig(OUT / "breakdown.png", dpi=150); plt.show()
by_group.to_csv(OUT / "breakdown.csv")

## §10 Interpretation

*Fill this in from the tables above after the GPU run. Answer each point with the measured numbers.*

1. **Which change gave the biggest gain?** Compare BM25 → tuned classical → zero-shot cross-encoder → fine-tuned cross-encoder → LoRA on nDCG@10.
2. **Did tuning the classical models matter?** BM25 default vs tuned; LogReg baseline vs tuned (best parameters are printed in §3 and §5).
3. **What did transfer learning give without any training?** Zero-shot cross-encoder vs the classical models. Does it differ between Russian (seen in pretraining) and Kazakh (not seen)?
4. **Frozen encoder vs full fine-tuning:** how much of the gain needs the encoder itself to adapt?
5. **LoRA vs full fine-tuning:** quality against trainable parameters and training time.
6. **Plain vs terminology questions:** hypothesis: neural models help most on plain-language questions, where the words differ from the statute's.
7. **Limits:** the questions are LLM-written, there is one labelled article per question, and the reranker cannot exceed the BM25 ceiling.
8. **Decision for the product:** which model becomes the reranker, given quality and CPU cost.

In [ ]:
# Reproducibility record + the files to bring back into the project
config = {"seed": SEED, "smoke": SMOKE, "device": DEVICE,
          "versions": {"python": sys.version.split()[0], "torch": torch.__version__, "transformers": transformers.__version__,
                       "peft": peft.__version__, "scikit-learn": sklearn.__version__, "bm25s": bm25s.__version__},
          "data": {"corpus": "AdiletCodex v1.0 (doi:10.5281/zenodo.22812626), 8 Tier-1 codes, RU + KK",
                   "train_pairs": len(train_pairs), "val_pairs": len(val_pairs), "test_pairs": len(test_pairs),
                   "test_questions": int(test_pairs["query_id"].nunique()), "candidates": K_CAND, "hard_negatives": N_NEG,
                   "ceiling": float(ceiling)},
          "bm25": {lang: {"k1": k1, "b": b} for lang, (k1, b) in best_bm25.items()},
          "logreg": lr_search.best_params_, "svm": svm_search.best_params_,
          "cross_encoder": {"base": CE_NAME, "frozen": FROZEN, "fine_tuning": FT, "lr_search": search},
          "lora": {"base": LLM_NAME, **LORA}}
(OUT / "config.json").write_text(json.dumps(config, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

import shutil
archive = shutil.make_archive("a3_results", "zip", OUT)
print("Saved:", archive, f"({os.path.getsize(archive) / 1e6:.0f} MB)")
print("It contains results.csv, breakdown.csv, config.json, the charts, the fine-tuned reranker and the LoRA adapter.")
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass